# NextLearn - Grade model (examGrade /20) on OULAD

Trains a `RandomForestRegressor` on real OU assessment outcomes and exports `rf-grade.joblib`.

**Label:** weight-weighted mean of all the student's assessment scores, scaled to /20 (the
outcome, so it uses full-course data unlike the pre-cutoff features). Students with no scored
assessment are dropped. Grade transfers less cleanly than risk - treat it as indicative.

In [ ]:
# Pin scikit-learn to the app's range so the joblib pickle loads in the ML service
# (ml/requirements.txt: scikit-learn>=1.7). If loading later warns about versions,
# match this to `python -c "import sklearn; print(sklearn.__version__)"` in the app env.
!pip install -q "scikit-learn>=1.7,<1.8" pandas numpy joblib


In [ ]:
import os, sys, joblib
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GroupKFold
from sklearn.metrics import roc_auc_score, mean_absolute_error

assert os.path.exists('oulad_features.py'), 'Upload ml/oulad/oulad_features.py next to this notebook'
from oulad_features import (build_dataset, to_training_csv,
                            FEATURES, RISK_LABEL, GRADE_LABEL, KEYS, DEMOGRAPHIC_COLS)

# Auto-locate the OULAD CSVs: ml/oulad/raw/ (Colab upload) or the repo's data/OULAD_DATASET/.
RAW_DIR = next((p for p in ['raw', '../../data/OULAD_DATASET', 'data/OULAD_DATASET']
                if os.path.exists(os.path.join(p, 'studentVle.csv'))), 'raw')
print('using RAW_DIR =', RAW_DIR)
MODELS_DIR = '../models' if os.path.isdir('../models') else '.'
DATA_DIR   = '../../data' if os.path.isdir('../../data') else '.'
SEED, CUTOFF_DAY = 42, 90
RF = dict(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)


In [ ]:
# include_demographics=True adds raw demographics for the ABLATION only
# (they never enter the deployed model).
df = build_dataset(raw_dir=RAW_DIR, cutoff_day=CUTOFF_DAY, include_demographics=True)
groups = df['id_student'].values   # for leakage-free, student-grouped CV
print('rows:', len(df), '| students:', df['id_student'].nunique())


In [ ]:
# --- EDA / validation ---
print('missingness (%):'); print((df[DEMOGRAPHIC_COLS].isna().mean()*100).round(1).to_string())
print('\ncorr of each feature with caughtUp (signs should be pedagogically sensible):')
print(df[FEATURES + [RISK_LABEL]].corr(numeric_only=True)[RISK_LABEL].drop(RISK_LABEL).sort_values().round(3).to_string())
df[FEATURES].describe().T.round(3)


In [ ]:
# --- Keep rows with a grade; train + leakage-free (student-grouped) CV ---
g = df.dropna(subset=[GRADE_LABEL]).copy()
print(f'rows with a grade: {len(g)} / {len(df)}')
Xg, yg, gg = g[FEATURES].astype(float).values, g[GRADE_LABEL].astype(float).values, g['id_student'].values
Xtr, Xte, ytr, yte = train_test_split(Xg, yg, test_size=0.2, random_state=SEED)
reg = RandomForestRegressor(**RF).fit(Xtr, ytr)
print(f'held-out          : MAE {mean_absolute_error(yte, reg.predict(Xte)):.3f} /20')
cvp = cross_val_predict(reg, Xg, yg, cv=GroupKFold(5), groups=gg, n_jobs=-1)
print(f'student-grouped CV: MAE {mean_absolute_error(yg, cvp):.3f} /20  <- leakage-free')


In [ ]:
# --- Refit on all data, export ---
reg.fit(Xg, yg)
grade_path = os.path.join(MODELS_DIR, 'rf-grade.joblib'); joblib.dump(reg, grade_path)
print('saved:', grade_path)


In [ ]:
# --- Smoke test ---
m = joblib.load(grade_path); row = g[FEATURES].iloc[[0]].astype(float).values
assert row.shape[1] == 9
print('predicted grade /20 row0:', float(m.predict(row)[0]))
print('OK - copy rf-grade.joblib into ml/models/')
